# PRACTICE 2: PREDICTING PRODUCT SALES
## CHẶNG 2: KỸ THUẬT ĐẶC TRƯNG & THIẾT LẬP PIPELINE (FEATURE ENGINEERING & PIPELINE)

> **Mục tiêu kỹ thuật:**
> - Trích xuất các đặc trưng miền thương mại điện tử: chuỗi thời gian mùa vụ, gom nhóm địa phương Top 10 bang + Other, cờ nhị phân khuyến mãi và khách sỉ B2B.
> - Kỹ thuật Cô Tiên: Kiểm tra và bóc tách các cặp đặc trưng đa cộng tuyến ($|r| > 0.6$).
> - Thiết lập hàng rào chống rò rỉ dữ liệu (Anti-Leakage Barrier): Phân chia Train/Test 80/20 (82.281 train / 20.571 test) trước khi fit ColumnTransformer.
> - Xuất các tệp `train.csv`, `test.csv`, `train_test_data.joblib` và `preprocessor.joblib`.

### 1. Trích xuất Đặc trưng Miền (Domain Feature Engineering) — Trương Thành Công
> **Nhiệm vụ Trương Thành Công:** Hoàn thiện hàm `extract_domain_features` trong `scripts/feature_engineering.py`.
- Trích xuất yếu tố thời gian từ `Date`: `month`, `day_of_week`, `is_weekend`.
- Gom nhóm địa lý `ship-state`: Top 10 bang lớn + `'Other'`.
- Tạo cờ nhị phân: `is_promoted` (từ `promotion-ids`) và `is_b2b` (từ `B2B`).

In [ ]:
# TODO: [Trương Thành Công] Gọi hàm extract_domain_features để trích xuất mùa vụ, cờ khuyến mãi, khách sỉ và top 10 bang
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from scripts.feature_engineering import (
    extract_domain_features,
    detect_collinear_features,
    split_train_test,
    build_column_transformer,
    export_train_test_csv
)

CLEAN_CSV = BASE_DIR / 'data' / 'processed' / 'clean_amazon_sales.csv'
df_clean = pd.read_csv(CLEAN_CSV)
df_feat = extract_domain_features(df_clean)
print(f"✓ Kích thước dữ liệu sau khi trích xuất đặc trưng: {df_feat.shape[0]:,} dòng, {df_feat.shape[1]} cột")
display(df_feat[['Date', 'month', 'day_of_week', 'is_weekend', 'is_promoted', 'is_b2b', 'state_grouped']].head(3))

### 2. Kỹ thuật Cô Tiên — Quét và Loại bỏ Đa cộng tuyến (|r| > 0.6) — Võ Thanh Phú
> **Nhiệm vụ Võ Thanh Phú:** Hoàn thiện hàm `detect_collinear_features` trong `scripts/feature_engineering.py`.
- Đo lường ma trận tương quan Pearson trên các thuộc tính số học.
- Phát hiện hiện tượng phụ thuộc tuyến tính cao để đảm bảo tính ổn định số học cho mô hình hồi quy.

In [ ]:
# TODO: [Võ Thanh Phú] Quét ma trận tương quan Pearson và phát hiện các cặp đặc trưng đa cộng tuyến (|r| > 0.6)
# Quét ma trận tương quan Pearson giữa các đặc trưng số
num_candidates = ['Qty', 'month', 'day_of_week', 'is_weekend', 'is_promoted', 'is_b2b']
corr_matrix = df_feat[num_candidates].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='Blues', linewidths=0.5)
plt.title('Ma trận tương quan Pearson giữa các biến số độc lập', fontweight='bold')
plt.show()

collinear_pairs = detect_collinear_features(df_feat[num_candidates], threshold=0.6)
print(f"• Các cặp đặc trưng đa cộng tuyến vượt ngưỡng (|r| > 0.6): {collinear_pairs}")
print("• Quyết định kỹ thuật: Loại bỏ 'is_weekend' do có tương quan cao với 'day_of_week' (r = 0.801).")

### 3. Phân chia Train/Test 80/20 & Đóng gói ColumnTransformer — Võ Thanh Phú
> **Nhiệm vụ Võ Thanh Phú:** Hoàn thiện các hàm `split_train_test`, `build_column_transformer`, `export_train_test_csv` trong `scripts/feature_engineering.py`.
- Hàng rào chống rò rỉ dữ liệu (Anti-Leakage Barrier): chia 82.281 train và 20.571 test.
- Target: y = np.log1p(Amount).
- Chuẩn hóa StandardScaler cho biến số và OneHotEncoder cho biến danh mục (CHỈ FIT TRÊN TRAIN).

In [ ]:
# TODO: [Võ Thanh Phú] Phân chia Train/Test 80/20 và fit ColumnTransformer trên tập Train
X_train, X_test, y_train, y_test, y_train_raw, y_test_raw = split_train_test(
    df_feat, test_size=0.2, random_state=42
)
print(f"✓ Tập huấn luyện (Train): {len(X_train):,} mẫu (80%)")
print(f"✓ Tập kiểm thử (Test)  : {len(X_test):,} mẫu (20%)")

num_cols = ['Qty', 'month', 'day_of_week', 'is_promoted', 'is_b2b']
cat_cols = ['Category', 'Size', 'Fulfilment', 'ship-service-level', 'Courier Status', 'state_grouped']

preprocessor = build_column_transformer(num_cols, cat_cols)
preprocessor.fit(X_train)

X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)
print(f"✓ Kích thước ma trận đặc trưng sau tiền xử lý: Train={X_train_trans.shape}, Test={X_test_trans.shape}")

In [ ]:
# TODO: [Võ Thanh Phú] Xuất train.csv, test.csv và lưu trữ train_test_data.joblib & preprocessor.joblib
# Xuất train.csv, test.csv và lưu trữ nhị phân
export_train_test_csv(X_train, X_test, y_train_raw, y_test_raw, BASE_DIR / 'data' / 'processed')

data_artifact_path = BASE_DIR / 'data' / 'processed' / 'train_test_data.joblib'
prep_artifact_path = BASE_DIR / 'models' / 'saved_models' / 'preprocessor.joblib'
prep_artifact_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump({
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'y_train_raw': y_train_raw,
    'y_test_raw': y_test_raw,
    'num_cols': num_cols,
    'cat_cols': cat_cols
}, data_artifact_path)

joblib.dump(preprocessor, prep_artifact_path)
print(f"✓ Đã xuất 'train.csv', 'test.csv' và lưu '{data_artifact_path.name}', '{prep_artifact_path.name}' thành công.")